# Two-Stage Detection: Faster R-CNN with torchvision

Fine-tune a two-stage, anchor-based detector ([Faster R-CNN](https://arxiv.org/abs/1506.01497)) on PASCAL VOC 2007.
Stage one, the **Region Proposal Network (RPN)**, scores a dense set of anchor boxes for "objectness" and refines
them into roughly a thousand class-agnostic proposals. Stage two, the **RoI heads**, pools a fixed-size feature for
each proposal with RoIAlign, then classifies it and refines its box again.

The model, matching, sampling, losses and post-processing all come from `torchvision.models.detection`; no
forward or backward pass is written by hand. The pieces that are still implemented in a few lines are the ones a CV
engineer should know cold: IoU, the R-CNN box-delta parameterisation, and the FPN level-assignment rule. Each one is
checked against torchvision. Everything else is inspected by calling torchvision's own components
(`AnchorGenerator`, `Matcher`, `BalancedPositiveNegativeSampler`, `MultiScaleRoIAlign`) on real data.

| Original assignment (hand-written) | This notebook |
|---|---|
| `VOC2007DetectionTiny`, padded `(B, 40, 5)` box tensor | `torchvision.datasets.VOCDetection` + `transforms.v2` + `tv_tensors.BoundingBoxes` |
| `DetectorBackboneWithFPN` | `fasterrcnn_resnet50_fpn_v2` backbone, or `create_feature_extractor` + `FeaturePyramidNetwork` |
| `generate_fpn_anchors` | `torchvision.models.detection.anchor_utils.AnchorGenerator` |
| `iou` | `torchvision.ops.box_iou` (reference IoU kept for understanding) |
| `rcnn_match_anchors_to_gt` | `Matcher` (inside `model.rpn` / `model.roi_heads`) |
| `rcnn_get_deltas_from_anchors`, `rcnn_apply_deltas_to_anchors` | `BoxCoder` (reference encode/decode kept) |
| `sample_rpn_training` | `BalancedPositiveNegativeSampler` |
| `RPNPredictionNetwork`, `RPN`, `FasterRCNN` classes | `torchvision.models.detection.FasterRCNN` (`RPNHead`, `RoIHeads`) |
| `reassign_proposals_to_fpn_levels` + `roi_align` | `torchvision.ops.MultiScaleRoIAlign` |
| `train_detector` helper | plain PyTorch loop: SGD, warmup + cosine LR, AMP, grad clipping |
| Cartucho mAP repo | `torchmetrics.detection.MeanAveragePrecision` |

One functional upgrade: the original second stage only classified proposals. torchvision's RoI heads also regress
per-class box refinements, which is standard Faster R-CNN.

No local `.py` files are needed; the notebook runs on its own (Colab or local, GPU strongly recommended).

## Setup

In [ ]:
# torchvision >= 0.16 is needed for transforms.v2 / tv_tensors.
%pip install -q "torchvision>=0.16" torchmetrics pycocotools

In [ ]:
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchmetrics.detection import MeanAveragePrecision
from torchvision import tv_tensors
from torchvision.datasets import VOCDetection
from torchvision.ops import batched_nms, box_iou, nms, sigmoid_focal_loss
from torchvision.transforms import v2 as T
from torchvision.utils import draw_bounding_boxes

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 8)

print(f"torch {torch.__version__} | torchvision {torchvision.__version__}")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"  # mixed precision only on CUDA
print("device:", DEVICE, "| AMP:", USE_AMP)


def seed_everything(seed: int = 0):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(0)

### Configuration

`BACKBONE = "resnet50"` fine-tunes torchvision's COCO-pretrained Faster R-CNN v2 (ResNet-50 FPN) and replaces only
the final box predictor for the 20 VOC classes; this is the standard recipe. `BACKBONE = "regnet_x_400mf"`
builds the small RegNet + FPN model from the original assignment out of library parts, with a custom anchor
generator and RoI pooler. It is much lighter but needs a longer schedule.

On Colab, point `DATA_ROOT` at Google Drive (after `drive.mount`) so the dataset survives runtime restarts.

In [ ]:
DATA_ROOT = Path("data")          # e.g. Path("/content/drive/MyDrive/datasets") on Colab
BACKBONE = "resnet50"             # "resnet50" (COCO-pretrained Faster R-CNN v2) | "regnet_x_400mf" (light)
MIN_SIZE, MAX_SIZE = 512, 853     # the model resizes the shorter side to MIN_SIZE (capped at MAX_SIZE)
BATCH_SIZE = 8                    # lower this if you run out of GPU memory
NUM_EPOCHS = 12                   # ~3.7k iterations on VOC07 train; use 30+ for the regnet backbone
LR = 0.01                         # for batch size 8; scale linearly with batch size
WEIGHT_DECAY = 1e-4
WARMUP_ITERS = 500
NUM_WORKERS = min(4, os.cpu_count() or 1)
CKPT_PATH = Path("checkpoints") / f"faster_rcnn_{BACKBONE}_voc07.pt"

## Data: PASCAL VOC 2007

`torchvision.datasets.VOCDetection` parses the XML annotations; a thin wrapper turns them into the format every
torchvision detector expects:

- **image**: `float32` tensor `(3, H, W)` in `[0, 1]`, any size. The model's own `GeneralizedRCNNTransform` handles
  resizing and ImageNet normalisation, so there is no manual `Normalize` / un-normalise step.
- **target**: `{"boxes": BoundingBoxes (N, 4) XYXY in absolute pixels, "labels": int64 (N,)}`, with label `0`
  reserved for background, so VOC classes are `1..20`.

Batches are *lists* of images and targets, so images of different sizes and different box counts need no padding
(this replaces the old `(B, 40, 5)` tensor padded with `-1`). Boxes flagged *difficult* are dropped, as in the
standard VOC protocol. `transforms.v2` augments the image and its boxes together.

If the official VOC server is slow or down, download the tar manually and extract it under `DATA_ROOT`:

```bash
wget https://web.eecs.umich.edu/~justincj/data/VOCtrainval_06-Nov-2007.tar
tar xf VOCtrainval_06-Nov-2007.tar -C data/   # gives data/VOCdevkit/VOC2007/...
```

In [ ]:
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person", "pottedplant", "sheep", "sofa",
    "train", "tvmonitor",
]
IDX_TO_NAME = ["__background__"] + VOC_CLASSES
CLASS_TO_IDX = {name: i for i, name in enumerate(IDX_TO_NAME)}
NUM_CLASSES = len(IDX_TO_NAME)  # 21: torchvision counts the background slot


class VOCDetectionV2(Dataset):
    """VOC2007 -> (image, {"boxes": BoundingBoxes[XYXY], "labels": int64}) for torchvision detectors."""

    def __init__(self, root, image_set, transforms=None, download=False):
        self.voc = VOCDetection(root, year="2007", image_set=image_set, download=download)
        self.transforms = transforms

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, ann = self.voc[idx]
        objects = [o for o in ann["annotation"]["object"] if o["difficult"] == "0"]
        boxes = torch.tensor(
            [[float(o["bndbox"][k]) for k in ("xmin", "ymin", "xmax", "ymax")] for o in objects],
            dtype=torch.float32,
        ).reshape(-1, 4)
        boxes[:, :2] -= 1  # VOC pixel indices are 1-based
        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(image.height, image.width)),
            "labels": torch.tensor([CLASS_TO_IDX[o["name"]] for o in objects], dtype=torch.int64),
        }
        if self.transforms is not None:
            image, target = self.transforms(image, target)
        return image, target


train_tf = T.Compose([
    T.ToImage(),
    T.RandomPhotometricDistort(p=0.5),
    T.RandomHorizontalFlip(p=0.5),
    T.ToDtype(torch.float32, scale=True),
    T.SanitizeBoundingBoxes(),  # drops boxes that became degenerate
])
eval_tf = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

download = not (DATA_ROOT / "VOCdevkit" / "VOC2007").exists()
train_ds = VOCDetectionV2(DATA_ROOT, "train", train_tf, download=download)
val_ds = VOCDetectionV2(DATA_ROOT, "val", eval_tf)
print(f"train: {len(train_ds)} images | val: {len(val_ds)} images")


def collate_fn(batch):
    return tuple(zip(*batch))  # (images, targets) as tuples, no padding


loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=DEVICE.type == "cuda",
    persistent_workers=NUM_WORKERS > 0,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

images, targets = next(iter(train_loader))
print(f"batch: {len(images)} images, first image {tuple(images[0].shape)}")
print("first target:", {k: (type(v).__name__, tuple(v.shape)) for k, v in targets[0].items()})

### Visualise a few training samples

In [ ]:
def to_uint8(image):
    return (image.detach().cpu().clamp(0, 1) * 255).to(torch.uint8)


def show(image, target=None, pred=None, score_thresh=0.5, ax=None, title=None):
    """Ground truth in green, predictions (score >= score_thresh) in red."""
    ax = ax or plt.gca()
    img = to_uint8(image)
    if target is not None and len(target["boxes"]):
        names = [IDX_TO_NAME[l] for l in target["labels"].tolist()]
        img = draw_bounding_boxes(img, target["boxes"].cpu(), labels=names, colors="lime", width=2)
    if pred is not None:
        keep = pred["scores"] >= score_thresh
        if keep.any():
            names = [f"{IDX_TO_NAME[l]} {s:.2f}"
                     for l, s in zip(pred["labels"][keep].tolist(), pred["scores"][keep].tolist())]
            img = draw_bounding_boxes(img, pred["boxes"][keep].cpu(), labels=names, colors="red", width=2)
    ax.imshow(img.permute(1, 2, 0))
    ax.axis("off")
    if title:
        ax.set_title(title)


fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, idx in zip(axes, range(3)):
    show(*train_ds[idx], ax=ax, title=f"train[{idx}]")
plt.tight_layout()
plt.show()

## Build the model

Two standard ways to adapt a torchvision detector to a new dataset:

1. **Fine-tune a pretrained detector.** Load COCO weights and replace `roi_heads.box_predictor`, the only
   class-dependent layer. The RPN is class-agnostic, so it keeps its COCO weights and produces good proposals
   from the first iteration.
2. **Plug in your own backbone.** Any module that returns an `OrderedDict` of feature maps and has
   `out_channels` works. You then pass an `AnchorGenerator` with one size tuple per feature map, and a
   `MultiScaleRoIAlign` that names the maps the second stage pools from.

In [ ]:
from torchvision.models import RegNet_X_400MF_Weights, regnet_x_400mf
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.ops import FeaturePyramidNetwork, FrozenBatchNorm2d


class RegNetFPN(nn.Module):
    """RegNetX-400MF trunk + torchvision FPN (the backbone from the original assignment, built from library parts).

    torchvision detectors accept any backbone that returns an OrderedDict of feature maps and exposes
    `out_channels`. BatchNorm is frozen because detection batches are too small for good BN statistics.
    """

    def __init__(self, out_channels=128, extra_blocks=None):
        super().__init__()
        trunk = regnet_x_400mf(weights=RegNet_X_400MF_Weights.IMAGENET1K_V2, norm_layer=FrozenBatchNorm2d)
        self.body = create_feature_extractor(
            trunk,
            return_nodes={"trunk_output.block2": "p3", "trunk_output.block3": "p4", "trunk_output.block4": "p5"},
        )
        with torch.no_grad():
            in_channels = [f.shape[1] for f in self.body(torch.zeros(1, 3, 64, 64)).values()]
        self.fpn = FeaturePyramidNetwork(in_channels, out_channels, extra_blocks=extra_blocks)
        self.out_channels = out_channels

    def forward(self, x):
        return self.fpn(self.body(x))

In [ ]:
from torchvision.models.detection import FasterRCNN, FasterRCNN_ResNet50_FPN_V2_Weights, fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import MultiScaleRoIAlign
from torchvision.ops.feature_pyramid_network import LastLevelMaxPool


def build_faster_rcnn(backbone=BACKBONE):
    if backbone == "resnet50":
        model = fasterrcnn_resnet50_fpn_v2(
            weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1, min_size=MIN_SIZE, max_size=MAX_SIZE
        )
        in_features = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    elif backbone == "regnet_x_400mf":
        fpn_backbone = RegNetFPN(out_channels=128, extra_blocks=LastLevelMaxPool())  # p3, p4, p5, pool
        anchor_generator = AnchorGenerator(
            sizes=((32,), (64,), (128,), (256,)),           # one tuple per feature map
            aspect_ratios=((0.5, 1.0, 2.0),) * 4,
        )
        roi_pooler = MultiScaleRoIAlign(featmap_names=["p3", "p4", "p5"], output_size=7, sampling_ratio=2)
        model = FasterRCNN(
            fpn_backbone,
            num_classes=NUM_CLASSES,
            rpn_anchor_generator=anchor_generator,
            box_roi_pool=roi_pooler,
            min_size=MIN_SIZE,
            max_size=MAX_SIZE,
        )
    else:
        raise ValueError(f"unknown backbone {backbone!r}")
    return model


model = build_faster_rcnn()
n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Faster R-CNN/{BACKBONE}: {n_params / 1e6:.1f}M params ({n_train / 1e6:.1f}M trainable)")
print("input transform:", model.transform)

### The knobs that define Faster R-CNN

All hyper-parameters live on the sub-modules, which is also where you change them.

In [ ]:
rpn, roi = model.rpn, model.roi_heads
print("RPN")
print(f"  anchor sizes / ratios     : {rpn.anchor_generator.sizes} / {rpn.anchor_generator.aspect_ratios[0]}")
print(f"  anchors per location      : {rpn.anchor_generator.num_anchors_per_location()}")
print(f"  matcher fg / bg IoU       : {rpn.proposal_matcher.high_threshold} / {rpn.proposal_matcher.low_threshold}")
print(f"  sampled anchors per image : {rpn.fg_bg_sampler.batch_size_per_image} "
      f"({rpn.fg_bg_sampler.positive_fraction:.0%} positive at most)")
print(f"  pre/post-NMS top-n (train): {rpn._pre_nms_top_n['training']} / {rpn._post_nms_top_n['training']}")
print(f"  pre/post-NMS top-n (test) : {rpn._pre_nms_top_n['testing']} / {rpn._post_nms_top_n['testing']}")
print(f"  NMS IoU                   : {rpn.nms_thresh}")
print(f"  box-coder weights         : {rpn.box_coder.weights}")
print("RoI heads")
print(f"  matcher fg / bg IoU       : {roi.proposal_matcher.high_threshold} / {roi.proposal_matcher.low_threshold}")
print(f"  sampled RoIs per image    : {roi.fg_bg_sampler.batch_size_per_image} "
      f"({roi.fg_bg_sampler.positive_fraction:.0%} positive at most)")
print(f"  RoI pooler                : {roi.box_roi_pool}")
print(f"  box-coder weights         : {roi.box_coder.weights}")
print(f"  test score / NMS / max det: {roi.score_thresh} / {roi.nms_thresh} / {roi.detections_per_img}")

## Core concept 1: anchors

The RPN places `len(sizes) × len(aspect_ratios)` anchors at every location of every pyramid level. With FPN,
each level gets a single size (small anchors on fine levels, large ones on coarse levels) and three aspect ratios.
The anchors are generated from the real feature-map shapes, so the same code works for any image size.

In [ ]:
model.eval()
image, target = val_ds[0]
with torch.no_grad():
    # model.transform resizes + normalises the image and rescales the GT boxes to match.
    image_list, (target_t,) = model.transform([image], [target])
    feats = model.backbone(image_list.tensors)
    anchors = model.rpn.anchor_generator(image_list, list(feats.values()))[0]

print(f"original image {tuple(image.shape[-2:])} -> network input {tuple(image_list.tensors.shape[-2:])}")
counts = [f.shape[-2] * f.shape[-1] * model.rpn.anchor_generator.num_anchors_per_location()[i]
          for i, f in enumerate(feats.values())]
for (name, f), n in zip(feats.items(), counts):
    print(f"  level {name:>4}: feature {tuple(f.shape[-2:])} -> {n} anchors")
print(f"  total: {anchors.shape[0]} anchors")
print("base anchors of the first level (centred at 0):\n", model.rpn.anchor_generator.cell_anchors[0])

mean = torch.tensor(model.transform.image_mean).view(3, 1, 1)
std = torch.tensor(model.transform.image_std).view(3, 1, 1)
net_image = image_list.tensors[0] * std + mean          # undo the model's normalisation for display

## Core concept 2: IoU, anchor matching and sampling

Each anchor is compared with every GT box by **IoU**. torchvision's `Matcher` then labels it:

- **positive** if its best IoU ≥ 0.7, plus, for every GT box, the anchor(s) with the highest IoU to it
  (`allow_low_quality_matches=True`, so every object gets at least one positive anchor),
- **negative** if its best IoU < 0.3,
- **ignored** in between.

Negatives vastly outnumber positives. Instead of focal loss, Faster R-CNN samples a fixed mini-batch of 256 anchors per image with at most 50% positives (`BalancedPositiveNegativeSampler`).
The second stage repeats the same match-and-sample procedure on proposals, with an IoU threshold of 0.5 and
512 RoIs per image.

In [ ]:
def iou(boxes1, boxes2):
    """Pairwise IoU between (N, 4) and (M, 4) XYXY boxes -> (N, M)."""
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    top_left = torch.max(boxes1[:, None, :2], boxes2[None, :, :2])
    bottom_right = torch.min(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (bottom_right - top_left).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]
    return inter / (area1[:, None] + area2[None, :] - inter)


xy = torch.rand(200, 2) * 300
random_boxes = torch.cat([xy, xy + torch.rand(200, 2) * 100 + 1], dim=1)
torch.testing.assert_close(iou(random_boxes[:50], random_boxes), box_iou(random_boxes[:50], random_boxes))
print("IoU matches torchvision.ops.box_iou")

In [ ]:
from torchvision.models.detection._utils import Matcher

gt_boxes = target_t["boxes"].as_subclass(torch.Tensor)   # GT in network-input coordinates
iou_matrix = box_iou(gt_boxes, anchors)                    # (num_gt, num_anchors)
matched_idx = model.rpn.proposal_matcher(iou_matrix)       # GT index, or -1 (negative) / -2 (ignored)

labels = (matched_idx >= 0).float()
labels[matched_idx == Matcher.BELOW_LOW_THRESHOLD] = 0.0
labels[matched_idx == Matcher.BETWEEN_THRESHOLDS] = -1.0
print(f"positive {int((labels == 1).sum())} | negative {int((labels == 0).sum())} | "
      f"ignored {int((labels == -1).sum())}  (of {len(labels)} anchors)")

pos_mask, neg_mask = model.rpn.fg_bg_sampler([labels])
pos_mask, neg_mask = pos_mask[0].bool(), neg_mask[0].bool()
print(f"sampled for the RPN loss: {int(pos_mask.sum())} positive + {int(neg_mask.sum())} negative")

vis = draw_bounding_boxes(to_uint8(net_image), gt_boxes, colors="lime", width=3)
vis = draw_bounding_boxes(vis, anchors[pos_mask], colors="red", width=1)
plt.imshow(vis.permute(1, 2, 0))
plt.title("GT boxes (green) and sampled positive anchors (red)")
plt.axis("off")
plt.show()

## Core concept 3: box-delta parameterisation

Unlike FCOS's edge distances, R-CNN regresses a box relative to its anchor (or, in stage two, its proposal)
as a centre shift scaled by the anchor size plus a log-space scale change:

$$t_x = w_x\frac{x - x_a}{w_a},\quad t_y = w_y\frac{y - y_a}{h_a},\quad t_w = w_w\log\frac{w}{w_a},\quad t_h = w_h\log\frac{h}{h_a}$$

The weights $(w_x, w_y, w_w, w_h)$ rescale the targets to unit-ish variance: `(1, 1, 1, 1)` in the RPN and
`(10, 10, 5, 5)` in the RoI heads (printed above). torchvision implements this as `BoxCoder`.

In [ ]:
def rcnn_encode(anchors, gt, weights=(1.0, 1.0, 1.0, 1.0)):
    wx, wy, ww, wh = weights
    aw, ah = anchors[:, 2] - anchors[:, 0], anchors[:, 3] - anchors[:, 1]
    ax, ay = anchors[:, 0] + 0.5 * aw, anchors[:, 1] + 0.5 * ah
    gw, gh = gt[:, 2] - gt[:, 0], gt[:, 3] - gt[:, 1]
    gx, gy = gt[:, 0] + 0.5 * gw, gt[:, 1] + 0.5 * gh
    return torch.stack([wx * (gx - ax) / aw, wy * (gy - ay) / ah,
                        ww * torch.log(gw / aw), wh * torch.log(gh / ah)], dim=1)


def rcnn_decode(deltas, anchors, weights=(1.0, 1.0, 1.0, 1.0)):
    wx, wy, ww, wh = weights
    aw, ah = anchors[:, 2] - anchors[:, 0], anchors[:, 3] - anchors[:, 1]
    ax, ay = anchors[:, 0] + 0.5 * aw, anchors[:, 1] + 0.5 * ah
    x, y = ax + deltas[:, 0] / wx * aw, ay + deltas[:, 1] / wy * ah
    w, h = aw * torch.exp(deltas[:, 2] / ww), ah * torch.exp(deltas[:, 3] / wh)
    return torch.stack([x - 0.5 * w, y - 0.5 * h, x + 0.5 * w, y + 0.5 * h], dim=1)


from torchvision.models.detection._utils import BoxCoder

toy_anchors = torch.tensor([[20.0, 40, 80, 90], [10, 10, 50, 50], [120, 100, 200, 200]])
toy_gt = torch.tensor([[10.0, 15, 100, 115], [30, 20, 40, 30], [120, 100, 200, 200]])
for weights in [(1.0, 1.0, 1.0, 1.0), (10.0, 10.0, 5.0, 5.0)]:
    deltas = rcnn_encode(toy_anchors, toy_gt, weights)
    coder = BoxCoder(weights)
    torch.testing.assert_close(rcnn_decode(deltas, toy_anchors, weights), toy_gt)
    torch.testing.assert_close(coder.encode_single(toy_gt, toy_anchors), deltas)
    torch.testing.assert_close(coder.decode_single(deltas, toy_anchors), toy_gt)
print("encode/decode match torchvision's BoxCoder for RPN and RoI-head weights")

## Core concept 4: RoIAlign and FPN level assignment

The second stage needs a fixed-size feature for each proposal regardless of its size. Each proposal is first assigned
to one pyramid level by its scale, following Eq. (1) of the [FPN paper](https://arxiv.org/abs/1612.03144):

$$k = \left\lfloor k_0 + \log_2\left(\sqrt{wh} / 224\right) \right\rfloor,\qquad k_0 = 4$$

clamped to the available levels, so a 224×224 proposal pools from P4, half that size from P3, and so on.
RoIAlign then bilinearly samples a 7×7 grid from that level (no coordinate rounding, unlike RoIPool).
`MultiScaleRoIAlign` does both steps; the two-line formula is checked against its internal level mapper.

In [ ]:
def fpn_level(boxes, k_min, k_max, k0=4, canonical_size=224, eps=1e-6):
    scale = torch.sqrt((boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]))
    return torch.floor(k0 + torch.log2(scale / canonical_size) + eps).clamp(k_min, k_max).long()


with torch.no_grad():
    proposals, _ = model.rpn(image_list, feats)   # eval mode: top-1000 proposals after NMS
    roi_feats = model.roi_heads.box_roi_pool(feats, proposals, image_list.image_sizes)
print(f"{proposals[0].shape[0]} proposals -> pooled RoI features {tuple(roi_feats.shape)}")

mapper = model.roi_heads.box_roi_pool.map_levels    # set up during the first forward pass
levels = fpn_level(proposals[0], mapper.k_min, mapper.k_max)
torch.testing.assert_close(levels - mapper.k_min, mapper(proposals))
print("level assignment matches MultiScaleRoIAlign;",
      {f"P{k}": int((levels == k).sum()) for k in range(mapper.k_min, mapper.k_max + 1)})

## Training and evaluation utilities

A plain PyTorch loop: torchvision detectors return a **loss dict** in `train()` mode and a **list of prediction
dicts** in `eval()` mode, so the loop only sums the losses and calls `backward()`. Standard detection recipe:
SGD with momentum, linear warmup then cosine decay, AMP, gradient clipping.

Evaluation uses `torchmetrics.detection.MeanAveragePrecision` (COCO-style, pycocotools backend) twice:
once at IoU 0.5 only (the VOC metric) and once over IoU 0.50:0.95 (the COCO metric). Note that torchmetrics uses
101-point interpolation, so VOC numbers differ slightly from the 11-point VOC2007 devkit.

In [ ]:
def to_device(images, targets, device):
    images = [img.to(device, non_blocking=True) for img in images]
    targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]
    return images, targets


def train_detector(model, loader, *, num_epochs=None, max_iters=None, lr=LR, weight_decay=WEIGHT_DECAY,
                   warmup_iters=WARMUP_ITERS, grad_clip=10.0, device=DEVICE, use_amp=USE_AMP, log_every=50):
    total_iters = max_iters if max_iters is not None else num_epochs * len(loader)
    warmup_iters = max(1, min(warmup_iters, total_iters // 2))

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1e-3, total_iters=warmup_iters),
            torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_iters - warmup_iters)),
        ],
        milestones=[warmup_iters],
    )
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    history, it, start = [], 0, time.time()
    model.train()
    while it < total_iters:
        for images, targets in loader:
            images, targets = to_device(images, targets, device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss_dict = model(images, targets)
                loss = sum(loss_dict.values())
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss at iter {it}: {loss_dict}")

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            it += 1
            history.append({"loss": loss.item(), **{k: v.item() for k, v in loss_dict.items()},
                            "lr": optimizer.param_groups[0]["lr"]})
            if it == 1 or it % log_every == 0 or it == total_iters:
                h = history[-1]
                parts = " ".join(f"{k}={v:.3f}" for k, v in h.items() if k not in ("loss", "lr"))
                print(f"[{it:>5}/{total_iters}] loss={h['loss']:.3f} | {parts} | "
                      f"lr={h['lr']:.2e} | {time.time() - start:.0f}s")
            if it >= total_iters:
                break
    return history


def plot_history(history, smooth=20):
    fig, ax = plt.subplots(figsize=(10, 4))
    for key in [k for k in history[0] if k != "lr"]:
        values = torch.tensor([h[key] for h in history])
        if len(values) >= smooth:
            values = values.unfold(0, smooth, 1).mean(1)
        ax.plot(values, label=key)
    ax.set_xlabel("iteration")
    ax.set_ylabel("loss")
    ax.set_title(f"Training losses (moving average over {smooth} iters)")
    ax.legend()
    plt.show()


@torch.no_grad()
def evaluate(model, loader, device=DEVICE, max_batches=None):
    model.eval()
    voc_map = MeanAveragePrecision(iou_type="bbox", iou_thresholds=[0.5], class_metrics=True)
    coco_map = MeanAveragePrecision(iou_type="bbox")
    for i, (images, targets) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        outputs = model([img.to(device) for img in images])
        preds = [{k: v.cpu() for k, v in o.items()} for o in outputs]
        gts = [{"boxes": t["boxes"].as_subclass(torch.Tensor), "labels": t["labels"]} for t in targets]
        voc_map.update(preds, gts)
        coco_map.update(preds, gts)
    return voc_map.compute(), coco_map.compute()


def report(voc_res, coco_res):
    print(f"VOC  mAP@0.5       : {voc_res['map'].item():.3f}")
    print(f"COCO mAP@[.50:.95] : {coco_res['map'].item():.3f} "
          f"(AP50 {coco_res['map_50'].item():.3f}, AP75 {coco_res['map_75'].item():.3f}, "
          f"AP_small {coco_res['map_small'].item():.3f})")
    per_class = sorted(
        zip([IDX_TO_NAME[c] for c in voc_res["classes"].tolist()], voc_res["map_per_class"].tolist()),
        key=lambda kv: kv[1],
    )
    names, aps = zip(*per_class)
    plt.figure(figsize=(8, 7))
    plt.barh(names, aps)
    plt.xlabel("AP@0.5")
    plt.title("Per-class AP on VOC2007 val")
    plt.tight_layout()
    plt.show()

## The torchvision detection API contract

One forward pass in each mode shows everything the training loop and evaluator rely on.

Faster R-CNN returns two RPN losses and two RoI-head losses.

In [ ]:
model = model.to(DEVICE)
images, targets = to_device(*next(iter(train_loader)), DEVICE)

model.train()
loss_dict = model(images, targets)
print("train mode ->", {k: round(v.item(), 4) for k, v in loss_dict.items()})

model.eval()
with torch.no_grad():
    outputs = model(images[:2])
print("eval mode  ->", [{k: tuple(v.shape) for k, v in o.items()} for o in outputs])

## Sanity check: overfit 16 images

Before spending GPU time on the full run, make sure the loss collapses on a tiny subset (no augmentation).
If it does not, something is wrong with the data or the model setup.

In [ ]:
seed_everything(0)
overfit_ds = Subset(VOCDetectionV2(DATA_ROOT, "train", eval_tf), range(16))
overfit_loader = DataLoader(overfit_ds, batch_size=4, shuffle=True, collate_fn=collate_fn)

sanity_model = build_faster_rcnn().to(DEVICE)
sanity_history = train_detector(sanity_model, overfit_loader, max_iters=300, warmup_iters=50, log_every=50)
plot_history(sanity_history)

sanity_model.eval()
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
with torch.no_grad():
    for ax, (img, tgt) in zip(axes, [overfit_ds[i] for i in range(3)]):
        show(img, tgt, sanity_model([img.to(DEVICE)])[0], ax=ax)
plt.tight_layout()
plt.show()

del sanity_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## Train on VOC2007 train

With the defaults (ResNet-50 FPN v2, 512 px, batch 8, AMP) expect roughly an hour on a Colab T4; reduce
`NUM_EPOCHS` or switch to the RegNet backbone for a quicker run.

In [ ]:
seed_everything(0)
model = build_faster_rcnn().to(DEVICE)
history = train_detector(model, train_loader, num_epochs=NUM_EPOCHS, log_every=100)
plot_history(history)

CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), CKPT_PATH)
print("saved", CKPT_PATH)

In [ ]:
# Optional: reload the trained weights (e.g. in a fresh runtime) instead of re-training.
model = build_faster_rcnn()
model.load_state_dict(torch.load(CKPT_PATH, map_location="cpu", weights_only=True))
model = model.to(DEVICE).eval()

## Inference

Stage-two post-processing: per-class softmax scores, per-class box decoding, clipping, removal of tiny boxes,
`score_thresh` filtering, class-aware `batched_nms`, and the top `detections_per_img`. The thresholds are attributes
of `model.roi_heads`. The plots only draw boxes scoring at least 0.5.

In [ ]:
model.roi_heads.score_thresh = 0.05
model.roi_heads.nms_thresh = 0.5
model.eval()
idxs = torch.linspace(0, len(val_ds) - 1, steps=6).long().tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
with torch.no_grad():
    for ax, idx in zip(axes.flat, idxs):
        img, tgt = val_ds[idx]
        show(img, tgt, model([img.to(DEVICE)])[0], score_thresh=0.5, ax=ax, title=f"val[{idx}]")
plt.tight_layout()
plt.show()

### Looking inside: stage-one proposals vs. final detections

Calling the sub-modules directly is the easiest way to debug a two-stage detector: if the right object is missing
from the proposals, the problem is in the RPN; if it is there but misclassified, it is in the RoI heads.

In [ ]:
model.eval()
image, target = val_ds[1]
with torch.no_grad():
    image_list, _ = model.transform([image.to(DEVICE)])
    feats = model.backbone(image_list.tensors)
    proposals, _ = model.rpn(image_list, feats)
    # map proposals from network-input coordinates back to the original image
    proposals = model.transform.postprocess(
        [{"boxes": proposals[0]}], image_list.image_sizes, [tuple(image.shape[-2:])]
    )[0]["boxes"]
    detections = model([image.to(DEVICE)])[0]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].imshow(draw_bounding_boxes(to_uint8(image), proposals[:30].cpu(), colors="cyan", width=1).permute(1, 2, 0))
axes[0].set_title(f"Top 30 of {len(proposals)} RPN proposals")
axes[0].axis("off")
show(image, target, detections, ax=axes[1], title="Final detections (score >= 0.5)")
plt.tight_layout()
plt.show()

## Evaluation on VOC2007 val

The original assignment's Faster R-CNN (small RegNet trained from scratch, no second-stage box regression) scored
well below its FCOS. A COCO-pretrained Faster R-CNN v2 fine-tuned with these settings should do much better.
Things to try for more: a larger `MIN_SIZE`, a longer schedule, training on VOC07+12 `trainval`, or stronger
augmentation (`T.RandomIoUCrop`, `T.RandomZoomOut`, multi-scale `min_size`).

In [ ]:
voc_res, coco_res = evaluate(model, val_loader)
report(voc_res, coco_res)